##### Drought Index Computation Workflow (SPI / SPEI-12) — Calibrate then Compute
Computes standardized precipitation indices from TerraClimate for the WIA
**hazard-impact** dimension, over the last calendar year.

Two-phase workflow (mirrors the "extract once, reuse" spirit):

1. **CALIBRATE** — fit the SPI/SPEI distribution parameters on the WMO 1991-2020
   climatology for this country's bounding box, and **save** them to NetCDF.
   Runs only the first time for a country (or when you force a refit); every
   later run reuses the saved parameters.
2. **COMPUTE** — load only the **last two calendar years** of TerraClimate
   (needed so the 12-month index at January of the target year has its full
   accumulation tail), apply the cached parameters, and write one index
   GeoTIFF per month of the target year, plus a minimal run log.

- Source: TerraClimate `ppt` (and `pet` for SPEI), per-year NetCDFs
- Indices: SPI-12 (gamma) and SPEI-12 (Pearson III) — 12-month scale, to match
  the exposure layer [`01_compute_SPI_SPEI.ipynb`](https://github.com/dohyung-kim/precip-index/blob/main/notebooks/01_compute_SPI_SPEI.ipynb)
- Output: index rasters only. Thresholds, per-pixel month-counts, population
  overlay, admin aggregation and plots are a separate post-processing notebook.

Design notes
- Only the shapefile **bounding box** is used.
- The 1991-2020 fit does **not** need recomputing every run: the fitted
  parameter arrays have shape `(12, lat, lon)` — the calibration *length* never
  enters their size, only their values: fitted once and cached (with period in filename).
- On load, the cache is verified against the current bounding box by comparing
  the stored `lat`/`lon` coordinates (a grid mismatch would otherwise only
  surface as an array-broadcast error deep in the compute), plus a cheap check
  that the file's recorded calibration years match this run.
- TerraClimate is downloaded/kept for the calibration period **and** the last two
  calendar years; TerraClimate files can be reused across countries.

Requirements: `xarray`, `netCDF4`, `scipy`, `numpy`, `geopandas`, `rasterio`,
`rioxarray`, `pycountry`, `python-dateutil` (CPU only — no GPU / Zarr / chunking).

**Acknowledgement**: This script reuses [precip-index repository](https://github.com/dohyung-kim/precip-index) from UNICEF climate data team.


In [ ]:
import json
import logging
import sys
import time
from datetime import date, datetime, timezone
from pathlib import Path
import urllib.request

from dateutil.relativedelta import relativedelta
import geopandas as gpd
import numpy as np
import pandas as pd
import xarray as xr
import pycountry

In [ ]:
# set up logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

##### 1.  IMPORT SPI/SPEI FROM THE [precip-index REPO](https://github.com/dohyung-kim/precip-index)
Reuse this repo's `spi()` / `spei()` (no reimplementation of the index maths).
The reuse pattern is the one advertised in their docstrings:
`spi(precip, scale=12, return_params=True)` to fit, then
`spi(new_precip, scale=12, fitting_params=params)` to apply.

In [ ]:
# Path to the precip-index src/ (edit if running from elsewhere).
# precip-index/src repo cloned in wia-scripts/precip-index/src
REPO_SRC = Path("./precip-index/src").resolve()
if not REPO_SRC.is_dir():
    raise FileNotFoundError(
        f"precip-index src/ not found at {REPO_SRC}. Set REPO_SRC accordingly."
    )
sys.path.insert(0, str(REPO_SRC))

from indices import spi, spei, save_fitting_params, load_fitting_params  # noqa: E402
from config import (  # noqa: E402
    DEFAULT_CALIBRATION_START_YEAR,
    DEFAULT_CALIBRATION_END_YEAR,
    Periodicity,
)
from utils import get_variable_name  # noqa: E402

log.info(f"Imported spi/spei from {REPO_SRC}")


##### 2.  CONFIGURATION  — edit this section for your run

In [ ]:
# -- Country for consultation --
country_list = [
    # "Kenya",
    # "Mali",
    # "Benin",
    # "Lebanon",
    # "Togo",
    # "Afghanistan",
    # "Ukraine",
    # "Burkina Faso",
    # "Niger",
    # "Honduras",
    # "El Salvador",
    # "Cameroon",
    # "Central African Republic",
    # "Myanmar",
    # "South Sudan",
    # "Syria",
    # "Ethiopia",
    # "Congo, The Democratic Republic of the",
    # "Haiti",
    "Somalia",
    # "Sudan",
    # "Yemen",
    # "Saint Vincent and the Grenadines",
    # "Grenada",
    # "Mozambique",
    # "State of Palestine",
]
country = country_list[0]

c_iso3 = (
    pycountry.countries.search_fuzzy(country)[0].alpha_3 if country != "Niger"
    # introduced Niger exception (known case to avoid Nigeria)
    else pycountry.countries.search_fuzzy(country)[1].alpha_3
)

# -- Boundary file for consultation --
# NOTE: output is raster, admin level used only for geojson id
admin_level = "2"

# geojson output file name
geo_filename = f"geojson_{c_iso3}_adm{admin_level}"
# Get geojson in shapes
geojson_file = [
    f.name
    for f in Path("./data_in/shapes/").iterdir()
    if f.name.endswith(".zip") and geo_filename in f.name
]

# geojson file exists error handling
if not geojson_file:
    raise FileNotFoundError(
        f"No admin file matching '{geo_filename}*.zip' in {Path("./data_in/shapes/")}"
    )

# read geojson into single country geometry gdf
gdf = gpd.read_file(
    "./data_in/shapes/" + f"{geojson_file[0]}"
).to_crs('EPSG:4326').dissolve()


In [ ]:
# -- Impact window: last completed calendar year (given TerraClimate updates) --
TARGET_YEAR = date.today().year - 1     # e.g. 2025 when run in 2026

# -- Index configuration --
SPEI_SCALE = 12                 # 12-month scale - match the exposure layer
RUN_SPI  = True                 # gamma-fitted SPI-12
RUN_SPEI = True                 # Pearson III-fitted SPEI-12
SPI_DISTRIBUTION  = "gamma"
SPEI_DISTRIBUTION = "pearson3"

# -- Calibration (standardization) period - WMO standard, matches exposure --
CALIB_START = DEFAULT_CALIBRATION_START_YEAR   # 1991
CALIB_END   = DEFAULT_CALIBRATION_END_YEAR     # 2020

# -- PET handling for SPEI (TerraClimate ships `pet`; no Thornthwaite) --
USE_TERRACLIMATE_PET = True

# -- Force a re-fit even if a valid cached calibration exists --
FORCE_RECALIBRATE = False

# -- TerraClimate download configuration --
TERRACLIMATE_BASE_URL = "https://climate.northwestknowledge.net/TERRACLIMATE-DATA"
TERRACLIMATE_MIN_SIZE = 10 * 1024 * 1024   # 10 MB — valid files are larger

log.info(
    f"{country} ({c_iso3}) | target year={TARGET_YEAR} | "
    f"SPI={RUN_SPI} SPEI={RUN_SPEI} scale={SPEI_SCALE} | "
    f"calib {CALIB_START}-{CALIB_END}"
)


In [ ]:
# -- Folders --
base_dir         = Path("./")
TERRACLIMATE_DIR = base_dir / "data_in" / "Raster_Datasets" / "TerraClimate"    # per-year NetCDFs
OUTPUT_DIR       = base_dir / "data_in" / "Raster_Datasets" / "Drought_Index"   # computed indexes from TerraClimate
PARAMS_DIR       = OUTPUT_DIR / "_fit_params"   # cached calibrations
OUTPUT_LOG_DIR   = base_dir / "data_out" / "logs"
for d in (OUTPUT_DIR, PARAMS_DIR, OUTPUT_LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

# The 12-month index at Jan of the target year needs the prior year's tail,
# so the COMPUTE phase loads exactly the last two calendar years.
COMPUTE_YEARS = [TARGET_YEAR - 1, TARGET_YEAR]

# bbox pad so country-edge pixels are fully covered
BBOX_PAD_DEG = 0.25


##### 3.  HELPERS

In [ ]:
def download_terraclimate_file(var: str, year: int, dest_dir: Path) -> Path:
    """Ensure TerraClimate_{var}_{year}.nc exists locally; download if missing.

    Mirrors the exposure notebook's fetch (same NKN source + size sanity check),
    minus the Colab/Google-Drive caching layer. Returns the local path.
    REF exposure notebook: notebooks/01_compute_SPI_SPEI.ipynb in precip-index repo.
    """
    dest_dir.mkdir(parents=True, exist_ok=True)
    fname = f"TerraClimate_{var}_{year}.nc"
    local = dest_dir / fname

    # Already present and non-truncated → reuse
    if local.exists() and local.stat().st_size >= TERRACLIMATE_MIN_SIZE:
        return local

    url = f"{TERRACLIMATE_BASE_URL}/{fname}"
    log.info(f"  downloading {fname} from NKN ...")
    try:
        urllib.request.urlretrieve(url, local)
    except Exception as e:
        if local.exists():
            local.unlink()   # remove partial file
        raise RuntimeError(f"Failed to download {fname} from {url}: {e}") from e

    if local.stat().st_size < TERRACLIMATE_MIN_SIZE:
        size = local.stat().st_size
        local.unlink()
        raise RuntimeError(
            f"Downloaded {fname} is only {size} bytes (< {TERRACLIMATE_MIN_SIZE}); "
            "treating as invalid."
        )
    log.info(f"  saved → {local}  ({local.stat().st_size/1e6:.0f} MB)")
    return local


In [ ]:
def load_terraclimate_var(var: str, years, bbox, auto_download: bool = True) -> xr.DataArray:
    """Open per-year TerraClimate files for `var`, subset to bbox, concat on time.

    `years` is an iterable of calendar years to load (need not be contiguous;
    calibration years and compute years are loaded separately).
    bbox = (lon_lo, lat_lo, lon_hi, lat_hi). TerraClimate latitude is descending,
    so lat is sliced as slice(north, south).
    """
    lon_lo, lat_lo, lon_hi, lat_hi = bbox
    das = []
    for year in sorted(set(int(y) for y in years)):
        fp = TERRACLIMATE_DIR / f"TerraClimate_{var}_{year}.nc"
        if not fp.exists():
            if auto_download:
                fp = download_terraclimate_file(var, year, TERRACLIMATE_DIR)
            else:
                raise FileNotFoundError(
                    f"Missing {fp.name} in {TERRACLIMATE_DIR} (auto_download=False)."
                )
        ds = xr.open_dataset(fp, decode_times=True)
        rename = {}
        if "latitude" in ds.dims:
            rename["latitude"] = "lat"
        if "longitude" in ds.dims:
            rename["longitude"] = "lon"
        if rename:
            ds = ds.rename(rename)
        da = ds[var].sel(lat=slice(lat_hi, lat_lo), lon=slice(lon_lo, lon_hi))
        das.append(da.load())
        ds.close()
    out = xr.concat(das, dim="time").sortby("time")
    return out.transpose("time", "lat", "lon")

In [ ]:
def assert_january_start(da: xr.DataArray, context: str) -> None:
    """Guard the monthly (years, 12) reshape assumption.

    The repo's reshape_to_2d folds the flat time axis into (n_years, 12) by
    position alone — column 0 is only 'January' if the array starts in January
    and spans whole calendar years. Nothing downstream checks this, so a window
    that accidentally starts mid-year would silently score, e.g., March data
    against January's fitted parameters. This makes that failure loud instead.
    """
    t = pd.to_datetime(da["time"].values)
    if len(t) == 0:
        raise RuntimeError(f"{context}: no time steps loaded.")
    if t[0].month != 1:
        raise RuntimeError(
            f"{context}: series starts in month {t[0].month} "
            f"({t[0].date()}), not January. The (years, 12) reshape needs a "
            "January start — check the loaded year range / date filtering."
        )
    if len(t) % 12 != 0:
        raise RuntimeError(
            f"{context}: {len(t)} months loaded — not a whole number of "
            "calendar years. Expected complete Jan–Dec years."
        )


In [ ]:
def params_path(index_kind: str) -> Path:
    """Cache path for the fitted distribution parameters of this country grid.

    The filename is the primary identifier: country, index, distribution, scale
    and calibration period. (The grid/bbox identity is verified on load by
    comparing stored coordinates — see load_calibration_checked.)
    """
    dist = SPI_DISTRIBUTION if index_kind == "spi" else SPEI_DISTRIBUTION
    return PARAMS_DIR / (
        f"{c_iso3}_{index_kind}_{dist}_{SPEI_SCALE}m_"
        f"calib{CALIB_START}_{CALIB_END}_params.nc"
    )

In [ ]:
def load_calibration_checked(index_kind, expected_lat, expected_lon):
    """Load cached fitting params, verifying grid + calibration identity.

    Returns the params dict if a valid cache exists and matches, else None.
    Guards two failure modes the library itself does not check:
      - grid/bbox mismatch (would otherwise surface as a broadcast error deep
        in the compute) — checked by comparing stored lat/lon coordinates.
      - wrong calibration window in a mislabelled file — checked against the
        file's recorded calibration-year attributes (the array shape alone
        cannot distinguish calibration periods).
    """
    dist = SPI_DISTRIBUTION if index_kind == "spi" else SPEI_DISTRIBUTION
    ppath = params_path(index_kind)
    if FORCE_RECALIBRATE or not ppath.exists():
        return None

    # Grid-identity check (coordinate values, not just shape)
    ds = xr.open_dataset(ppath)
    ok = True
    if "lat" in ds.coords and "lon" in ds.coords:
        clat, clon = ds["lat"].values, ds["lon"].values
        if (clat.shape != expected_lat.shape
                or not np.allclose(clat, expected_lat, atol=1e-6)):
            log.warning(f"  [{index_kind}] cached grid lat mismatch — ignoring cache")
            ok = False
        if (clon.shape != expected_lon.shape
                or not np.allclose(clon, expected_lon, atol=1e-6)):
            log.warning(f"  [{index_kind}] cached grid lon mismatch — ignoring cache")
            ok = False
    else:
        log.warning(f"  [{index_kind}] cache has no lat/lon coords — ignoring cache")
        ok = False

    # Calibration-year tripwire (cheap corroboration via global attrs)
    cs = str(ds.attrs.get("calibration_start_year", ""))
    ce = str(ds.attrs.get("calibration_end_year", ""))
    if cs and ce and (cs != str(CALIB_START) or ce != str(CALIB_END)):
        log.warning(
            f"  [{index_kind}] cached calibration {cs}-{ce} != requested "
            f"{CALIB_START}-{CALIB_END} — ignoring cache"
        )
        ok = False
    ds.close()

    if not ok:
        return None

    params = load_fitting_params(
        str(ppath), scale=SPEI_SCALE, periodicity="monthly", distribution=dist
    )
    log.info(f"  [{index_kind}] reusing cached calibration -> {ppath.name}")
    return params

In [ ]:
def calibrate_index(index_kind, ppt_cal, pet_cal):
    """Fit the distribution on the calibration-period array and cache params.

    Returns the params dict. The calibration array must start in January and
    span whole calendar years (CALIB_START..CALIB_END) so the internal
    (years, 12) reshape aligns calendar months correctly.
    """
    dist = SPI_DISTRIBUTION if index_kind == "spi" else SPEI_DISTRIBUTION
    ppath = params_path(index_kind)
    log.info(f"  [{index_kind}] fitting {dist} on {CALIB_START}-{CALIB_END} ...")

    if index_kind == "spi":
        _, params = spi(
            ppt_cal, scale=SPEI_SCALE,
            calibration_start_year=CALIB_START, calibration_end_year=CALIB_END,
            distribution=dist, return_params=True,
        )
    else:
        _, params = spei(
            ppt_cal, pet=pet_cal, scale=SPEI_SCALE,
            calibration_start_year=CALIB_START, calibration_end_year=CALIB_END,
            distribution=dist, return_params=True,
        )

    coords = {"lat": ppt_cal["lat"].values, "lon": ppt_cal["lon"].values}
    save_fitting_params(
        params, str(ppath), scale=SPEI_SCALE, periodicity="monthly",
        index_type=index_kind, calibration_start_year=CALIB_START,
        calibration_end_year=CALIB_END, coords=coords, distribution=dist,
    )
    log.info(f"  [{index_kind}] calibration saved -> {ppath.name}")
    return params


In [ ]:
def compute_window_index(index_kind, ppt_win, pet_win, params):
    """Apply cached params to the compute-window array; return index DataArray.

    ppt_win/pet_win span COMPUTE_YEARS (the last two calendar years), so the
    12-month accumulation is valid from January of the target year onward.
    """
    dist = SPI_DISTRIBUTION if index_kind == "spi" else SPEI_DISTRIBUTION
    if index_kind == "spi":
        da = spi(ppt_win, scale=SPEI_SCALE, distribution=dist, fitting_params=params)
    else:
        da = spei(ppt_win, pet=pet_win, scale=SPEI_SCALE,
                  distribution=dist, fitting_params=params)
    return da.transpose("time", "lat", "lon")


In [ ]:
def save_month_raster(da2d: xr.DataArray, out_path: Path, crs) -> float:
    """Write a single-band index GeoTIFF (float32) for one month. Returns size MB."""
    da2d = da2d.rio.write_crs(crs)
    da2d = da2d.rio.write_nodata(np.nan)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    da2d.rio.to_raster(str(out_path), driver="COG", compress="DEFLATE", dtype="float32")
    return out_path.stat().st_size / 1e6


##### 4.  COUNTRY BBOX  (only the bounding box is used)

In [ ]:
# country bbox from the dissolved geometry in gdf (CRS EPSG:4326)
w, s, e, n = gdf.total_bounds     # minx, miny, maxx, maxy

# pad bbox to ensure country-edge pixels are fully covered
bbox = (w - BBOX_PAD_DEG, s - BBOX_PAD_DEG, e + BBOX_PAD_DEG, n + BBOX_PAD_DEG)
log.info(f"Country bbox (padded): {bbox}")


##### 5.  PHASE A — CALIBRATE (fit once, reuse thereafter)
For each requested index, reuse the cached calibration if it exists and matches
this country's grid and calibration period; otherwise load the 1991-2020
TerraClimate for the bbox, fit, and cache. The compute grid (from the compute
window, Phase B) is used to validate the cache; here we load calibration data
lazily only when a fit is actually needed.

In [ ]:
run_t0 = time.time()
index_kinds = (["spi"] if RUN_SPI else []) + (["spei"] if RUN_SPEI else [])

# We need the target grid to check the cache. Load ONE compute-year month-slice
# cheaply to establish the exact (lat, lon) the compute will run on.
grid_probe = load_terraclimate_var("ppt", [TARGET_YEAR], bbox).isel(time=0)
grid_lat, grid_lon = grid_probe["lat"].values, grid_probe["lon"].values
log.info(f"Compute grid: {grid_lat.size} lat x {grid_lon.size} lon")

calib_meta = {}     # index_kind -> {"source": "cache"|"fitted", "n_calib_years": int}
params_by_kind = {}

# Lazily loaded calibration arrays (only if some index needs fitting)
_ppt_cal = _pet_cal = None

for kind in index_kinds:
    params = load_calibration_checked(kind, grid_lat, grid_lon)
    if params is not None:
        params_by_kind[kind] = params
        calib_meta[kind] = {"source": "cache", "n_calib_years": None}
        continue

    # Need to fit — load calibration-period TerraClimate for the bbox (once)
    if _ppt_cal is None:
        cal_years = list(range(CALIB_START, CALIB_END + 1))
        log.info(f"Loading TerraClimate ppt {CALIB_START}-{CALIB_END} for calibration ...")
        _ppt_cal = load_terraclimate_var("ppt", cal_years, bbox)
        assert_january_start(_ppt_cal, "calibration ppt")
        if RUN_SPEI and USE_TERRACLIMATE_PET:
            log.info(f"Loading TerraClimate pet {CALIB_START}-{CALIB_END} for calibration ...")
            _pet_cal = load_terraclimate_var("pet", cal_years, bbox)
            assert_january_start(_pet_cal, "calibration pet")
        # calibration array must align with the compute grid
        if (_ppt_cal["lat"].size != grid_lat.size
                or _ppt_cal["lon"].size != grid_lon.size):
            raise RuntimeError(
                "Calibration grid does not match compute grid — check the bbox/"
                "shapefile and TerraClimate consistency."
            )

    params = calibrate_index(kind, _ppt_cal, _pet_cal)
    params_by_kind[kind] = params
    n_cal_years = int(pd.to_datetime(_ppt_cal["time"].values).year.nunique())
    calib_meta[kind] = {"source": "fitted", "n_calib_years": n_cal_years}


##### 6.  PHASE B — COMPUTE the target-year index rasters
Load only the last two calendar years, apply the (cached or fresh) parameters,
keep the 12 months of the target year, and write one GeoTIFF per index per
month. Existing rasters are skipped (resumable).

In [ ]:
# Load the compute window: last two calendar years
log.info(f"Loading TerraClimate ppt {COMPUTE_YEARS} (compute window) ...")
ppt_win = load_terraclimate_var("ppt", COMPUTE_YEARS, bbox)
assert_january_start(ppt_win, "compute window ppt")
pet_win = None
if RUN_SPEI and USE_TERRACLIMATE_PET:
    log.info(f"Loading TerraClimate pet {COMPUTE_YEARS} (compute window) ...")
    pet_win = load_terraclimate_var("pet", COMPUTE_YEARS, bbox)
    assert_january_start(pet_win, "compute window pet")

native_crs = "EPSG:4326"    # TerraClimate is geographic
native_res = float(abs(ppt_win["lat"].values[1] - ppt_win["lat"].values[0]))

# Coverage sanity: need the full prior-year tail for Jan of the target year
t_win = pd.to_datetime(ppt_win["time"].values)
if t_win.min() > pd.Timestamp(f"{TARGET_YEAR}-01-01") - relativedelta(months=SPEI_SCALE):
    log.warning(
        "Compute window may be too short to accumulate SPEI-%d at %d-01. "
        "Ensure both %s are present.", SPEI_SCALE, TARGET_YEAR, COMPUTE_YEARS
    )


In [ ]:
rasters_manifest = []

for kind in index_kinds:
    dist = SPI_DISTRIBUTION if kind == "spi" else SPEI_DISTRIBUTION
    log.info(f"Computing {kind.upper()}-{SPEI_SCALE} ({dist}) for {TARGET_YEAR} ...")
    da_full = compute_window_index(kind, ppt_win, pet_win, params_by_kind[kind])

    # Keep only the target calendar year's 12 months
    t = pd.to_datetime(da_full["time"].values)
    keep = np.where(t.year == TARGET_YEAR)[0]
    if keep.size == 0:
        raise RuntimeError(f"No {kind} months fall in {TARGET_YEAR}.")
    da_win = da_full.isel(time=keep).sortby("time")

    var_label = get_variable_name(kind, SPEI_SCALE, Periodicity.monthly, distribution=dist)
    log.info(
        f"  {kind}: {da_win.sizes['time']} monthly rasters "
        f"({pd.to_datetime(da_win['time'].values[0]).date()} -> "
        f"{pd.to_datetime(da_win['time'].values[-1]).date()})"
    )

    for ti in range(da_win.sizes["time"]):
        da2d = da_win.isel(time=ti)
        month_label = pd.to_datetime(da2d["time"].values).strftime("%Y-%m")
        out_path = OUTPUT_DIR / f"{kind}{SPEI_SCALE}_{dist}_{c_iso3}_{month_label}.tif"
        if out_path.exists():
            size_mb = out_path.stat().st_size / 1e6
            log.info(f"    {out_path.name} exists ({size_mb:.2f} MB) - skipping")
            status = "cached"
        else:
            size_mb = save_month_raster(da2d, out_path, native_crs)
            log.info(f"    OK {out_path.name} ({size_mb:.2f} MB)")
            status = "written"
        rasters_manifest.append({
            "index": kind, "distribution": dist, "variable": var_label,
            "month": month_label, "file": out_path.name,
            "size_mb": round(size_mb, 3), "status": status,
        })


##### 7.  MINIMAL RUN LOG

In [ ]:
meta = {
    "pipeline": "drought_index_spi_spei",
    "created_at": datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S"),
    "iso3": c_iso3,
    "country": country,
    "target_year": TARGET_YEAR,
    "compute_years_loaded": COMPUTE_YEARS,
    "source": "TerraClimate (ppt, pet)",
    "scale_months": SPEI_SCALE,
    "indices": index_kinds,
    "spi_distribution": SPI_DISTRIBUTION if RUN_SPI else None,
    "spei_distribution": SPEI_DISTRIBUTION if RUN_SPEI else None,
    "calibration_start_year": CALIB_START,
    "calibration_end_year": CALIB_END,
    "calibration": calib_meta,           # per-index: cache vs fitted (+ n years)
    "bbox_pad_deg": BBOX_PAD_DEG,
    "bbox_epsg4326": {"west": bbox[0], "south": bbox[1],
                      "east": bbox[2], "north": bbox[3]},
    "native_crs": native_crs,
    "native_resolution_deg": native_res,
    "standardization_note": (
        "Distribution fitted once on the 1991-2020 climatology for this country "
        "grid and reused; target-year window standardized against that fit. "
        "Index rasters only - thresholds and aggregation are done in "
        "post-processing."
    ),
    "rasters": rasters_manifest,
    "summary": {
        "n_rasters_total": len(rasters_manifest),
        "n_written": sum(1 for r in rasters_manifest if r["status"] == "written"),
        "n_cached": sum(1 for r in rasters_manifest if r["status"] == "cached"),
        "months_per_index": len(rasters_manifest) // max(len(index_kinds), 1),
    },
    "total_runtime_s": round(time.time() - run_t0, 1),
    "completed_at": datetime.now(timezone.utc).isoformat(),
}
meta["run_id"] = (
    f"{meta['pipeline']}_{c_iso3}_{TARGET_YEAR}_{meta['created_at']}"
)

log_path = OUTPUT_LOG_DIR / f"{meta['run_id']}.json"
log_path.write_text(json.dumps(meta, indent=2, default=str))
log.info(f"Run log -> {log_path}")
log.info(
    f"Done in {meta['total_runtime_s']:.0f}s.  "
    f"{meta['summary']['n_written']} written, "
    f"{meta['summary']['n_cached']} cached."
)
log.info(f"Index rasters in: {OUTPUT_DIR.resolve()}")
log.info(f"Calibrations in: {PARAMS_DIR.resolve()}")
